In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
import h5py

In [ ]:
def calculate_csd(sim_path):
    """Calculates cloud statistics for a given simulation.

    Memory layout (original vs fixed):
      Original: cloud_areas + plume_areas + plume_mfs all live simultaneously
                → 3 × 23.9 GB + temps = ~78 GB → OOM on NC=190592 runs
      Fixed:    cloud_areas freed after attached_ind is known; plume arrays loaded
                for attached clouds only (~48K rows instead of 190K)
                → peak ~30 GB

    Also vectorizes the two Python loops from the original:
      mcbl_c/mctl_c:  NC-iteration loop    → np.argmax on summed area array
      calc_mean_mf:   NC×NT-iteration loop → advanced numpy indexing
    """

    dx, dy, dz = 250, 250, 50   # m
    grid_vol   = dx * 1e-3 * dy * 1e-3 * dz * 1e-3  # km³
    cbl_gap    = 3
    qclm_crit  = 1.0e-6

    sim_path = Path(sim_path)

    with open(sim_path / 'uninterrupted_large_clouds.json') as f:
        ul_clouds = json.load(f)
    ul_clouds_list = list(ul_clouds.keys())
    nc     = len(ul_clouds_list)
    id_list = np.asarray(ul_clouds_list, dtype=np.int32)

    # Phase 1: cloud area — full (NC, NT, NZ) load, freed once attached_ind is known
    with h5py.File(sim_path / 'hdf5/cloud_all_af.h5') as f:
        cloud_areas = f['af'][:]   # (NC, NT, NZ) float32

    nt = cloud_areas.shape[1]
    nz = cloud_areas.shape[2]

    cloud_volumes       = cloud_areas.sum(axis=2)
    cloud_times         = (cloud_volumes > 0).sum(axis=1)
    cloud_ini_time      = np.argmax(cloud_volumes > 0, axis=1)
    total_cloud_volumes = cloud_volumes.sum(axis=1) * grid_vol
    max_cloud_area      = cloud_areas.max(axis=(1, 2))
    cbl_c               = np.argmax(cloud_areas > 0, axis=2)

    # Vectorized min cloud base / max cloud top (replaces NC-iteration Python loop)
    area_z   = cloud_areas.sum(axis=1)   # (NC, NZ)
    has_area = area_z > 0
    mcbl_c   = np.argmax(has_area, axis=1)
    mctl_c   = (nz - 1) - np.argmax(has_area[:, ::-1], axis=1)
    del area_z, has_area, cloud_areas    # free ~23.9 GB

    with open(sim_path / 'pkl/qclm.pkl', 'rb') as f:
        qclm = np.asarray(pickle.load(f))
    mcbl = np.argmax(qclm > qclm_crit, axis=1)

    attached_c   = (cloud_volumes > 0) & (cbl_c - cbl_gap < mcbl[np.newaxis, :])
    del cbl_c
    attached_ind     = np.where(attached_c.any(axis=1))[0]
    attached_c_att   = attached_c[attached_ind, :]    # (n_att, NT)
    del attached_c, cloud_volumes

    # Phase 2: plume area — attached rows only (~0.6 GB instead of 23.9 GB)
    with h5py.File(sim_path / 'hdf5/plume_all_af.h5') as f:
        max_plume_area = f['af'][attached_ind].max(axis=(1, 2))

    # Phase 3: plume mass flux — attached rows only, vectorized (replaces NC×NT loop)
    with h5py.File(sim_path / 'hdf5/plume_all_mf.h5') as f:
        plume_mfs_att = f['mf'][attached_ind].astype(np.float64) * (dx * dy)

    t_idx = np.arange(nt)
    l_idx = np.clip(mcbl - 1, 0, nz - 3)
    mf_t  = (
        plume_mfs_att[:, t_idx, l_idx    ] +
        plume_mfs_att[:, t_idx, l_idx + 1] +
        plume_mfs_att[:, t_idx, l_idx + 2]
    ) / 3.0
    del plume_mfs_att

    attached_sum = attached_c_att.sum(axis=1).astype(np.float64)
    mean_cb_mf   = np.where(
        attached_sum > 0,
        (attached_c_att * mf_t).sum(axis=1) / np.maximum(attached_sum, 1),
        0.0,
    )
    del mf_t, attached_c_att

    mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)

    return (
        mean_cb_mf_c, mean_cb_mf,
        total_cloud_volumes[attached_ind], cloud_times[attached_ind],
        mcbl_c[attached_ind], mctl_c[attached_ind],
        cloud_ini_time[attached_ind], max_cloud_area[attached_ind],
        max_plume_area, attached_ind,
    )

In [ ]:
l_calc_csd = True
if l_calc_csd:
    case_name = 'goamazon_2pulse.largedom.r20251008.rerun'
    stat_ctl = calculate_csd(f'{case_name}/')
    with open(f'{case_name}/pkl/csd_stats.sparse.claude.pkl', 'wb') as f:
        pickle.dump(stat_ctl, f)
    case_name = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'
    stat_ehe1 = calculate_csd(f'{case_name}/')
    with open(f'{case_name}/pkl/csd_stats.sparse.claude.pkl', 'wb') as f:
        pickle.dump(stat_ehe1, f)

In [ ]:
ctl = 'goamazon_2pulse.largedom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'
with open(f'{ctl}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open(f'{ehe1}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ehe1 = pickle.load(f)

In [ ]:
nc_ehe1, = stat_ehe1[0].shape
nc_ctl, = stat_ctl[0].shape
print(f'{nc_ehe1} clouds in EHE1 simulation')
print(f'{nc_ctl} clouds in CTL simulation')